# Gate: 커스텀 도구로 구현하는 사람 개입 루프

많은 워크플로가 "완전 자동화"와 "항상 사람에게 묻기" 사이 어딘가에 놓여 있습니다. 경비 승인이 대표적입니다. 명확한 건은 에이전트가 알아서 처리하되, 애매한 건은 언제 사람 검토로 올려야 할지 알아야 합니다. 여기서는 보정이 중요합니다. 모든 것을 올려 보내는 에이전트는 함께 일하기 지치고, 아무것도 올리지 않는 에이전트는 위험합니다.

이 노트북에서는 두 개의 **커스텀 도구**를 중심으로 경비 승인 에이전트를 만듭니다. 명확한 건에는 `decide()`를, 애매한 건에는 `escalate()`를 씁니다. 둘 다 여러분의 애플리케이션을 왕복하며, 그 지점에서 결과를 기록하거나(decide) 검토자 앞에 올려놓게(escalate) 됩니다.

## 커스텀 도구란

지금까지 이 쿡북은 내장 `agent_toolset`(bash, read, write 등)을 사용했고, 이들은 모두 샌드박스 컨테이너 안에서 실행됩니다. **커스텀 도구**는 다릅니다. 에이전트가 이를 호출하면 세션이 멈추고 `agent.custom_tool_use` 이벤트를 내보냅니다. 여러분의 애플리케이션이 그 호출을 받아 원하는 코드를 실행한 뒤 `user.custom_tool_result` 이벤트를 POST로 돌려보냅니다. 그러면 그 결과가 에이전트의 컨텍스트에 담긴 채 세션이 재개됩니다.

다음 두 상황에 알맞은 형태입니다.

1. **데이터가 샌드박스에서 닿을 수 없는 곳에 있을 때.** 여러분의 네트워크 경계 안쪽에 있는 모든 것이 해당합니다. 에이전트는 왕복을 통해 여러분의 애플리케이션으로 되돌아옵니다.
2. **사람을 루프에 넣고 싶거나, 모든 호출 앞에 자체 감사·승인 계층을 두고 싶을 때.** 이 노트북이 하는 일이 그것입니다. `decide`와 `escalate`는 그저 추상적인 "도구"가 아니라, 여러분의 비즈니스 로직과 사람 검토자가 에이전트로부터 일을 넘겨받는 이음매입니다.

(다른 확장 패턴인 MCP 툴셋과 `resources=` 저장소 마운트는 각각 operate 노트북과 orchestrate 노트북에서 다룹니다.)

이 노트북은 두 부분으로 나뉩니다. A부에서는 이벤트를 로컬에서 스트리밍하며 커스텀 도구 호출이 도착할 때마다 응답해 세션을 구동합니다. 모든 것이 한 프로세스에서 일어나고 동작을 실시간으로 볼 수 있어 개발 중에 편리합니다. B부는 프로덕션 웹훅 패턴에 대한 짧은 안내이며, 전체 과정은 operate 노트북에서 처음부터 끝까지 다룹니다.

픽스처는 `example_data/gate/`에 있으며, `policy.yaml`과 정책의 모든 분기를 시험하는 영수증 열두 건이 들어 있습니다.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from anthropic import Anthropic
from utilities import wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()
FIXTURE = Path("example_data") / "gate"

## 1. 정책과 영수증 업로드

In [ ]:
policy = client.beta.files.upload(
    file=("policy.yaml", (FIXTURE / "policy.yaml").read_bytes(), "text/yaml")
)
receipts = client.beta.files.upload(
    file=(
        "receipts.jsonl",
        (FIXTURE / "inbox" / "receipts.jsonl").read_bytes(),
        "application/jsonl",
    )
)

## 2. 커스텀 도구 두 개를 갖춘 에이전트 정의하기

커스텀 도구는 내장 툴셋과 같은 `tools=` 배열에 `"type": "custom"`과 입력용 JSON 스키마를 함께 선언합니다. 각 선언은 그 도구가 무엇을 위한 것인지(`description`), 무엇을 인자로 호출해야 하는지(`input_schema`), 이름이 무엇인지를 모델에 알려 줍니다. 언제 호출할지는 에이전트가 정하고, 호출되었을 때 무엇을 할지는 여러분의 코드가 정합니다.

여기서는 내장 `agent_toolset_20260401`도 함께 켜 두어 에이전트가 정책 파일과 영수증을 직접 읽을 수 있게 합니다. `decide`와 `escalate`가 모든 판단을 왕복으로 만드는 두 커스텀 도구입니다.

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-gate",
    model=MODEL,
    system=(
        "You are an expense approver. Read each receipt in "
        "receipts.jsonl against the policy in policy.yaml and make "
        "exactly ONE tool call per receipt. Call decide(receipt_id, "
        "action, reason) for clear cases, or escalate(receipt_id, "
        "question) for ambiguous ones (near thresholds, unclear "
        "categories, suspicious notes). Once you've called decide "
        "or escalate for a given receipt, that receipt is finalized "
        "— do not call either tool for it again. After processing "
        "all receipts exactly once, stop."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        },
        {
            "type": "custom",
            "name": "decide",
            "description": "Record a final approve/reject for a clear-cut receipt.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "receipt_id": {"type": "string"},
                    "action": {"type": "string", "enum": ["approve", "reject"]},
                    "reason": {"type": "string"},
                },
                "required": ["receipt_id", "action", "reason"],
            },
        },
        {
            "type": "custom",
            "name": "escalate",
            "description": "Surface an ambiguous receipt for human review.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "receipt_id": {"type": "string"},
                    "question": {"type": "string"},
                },
                "required": ["receipt_id", "question"],
            },
        },
    ],
)

env = client.beta.environments.create(
    name="cookbook-gate-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {"type": "file", "file_id": policy.id, "mount_path": "policy.yaml"},
        {"type": "file", "file_id": receipts.id, "mount_path": "receipts.jsonl"},
    ],
    title="Expense gate",
)
print(f"session: {session.id}")

## A부: 개발 중 로컬 스트리밍

커스텀 도구 에이전트를 구동하는 가장 단순한 방법은 세션의 이벤트를 스트리밍하며 도구 호출이 도착할 때마다 반응하는 것입니다. `decide` 호출은 기록하고, `escalate` 호출에는 사람의 판단을 흉내 낸 응답을 즉석에서 줍니다. 실제 프로덕션에서는 에스컬레이션을 큐에 넣고 나중에 실제 검토자가 처리하게 하는데, 그 부분은 operate 노트북에서 다룹니다.

In [ ]:
def simulate_human_review(receipt_id: str, question: str) -> str:
    # Real implementation would show this in a UI and await input.
    # Here: reject anything the agent flags as suspicious.
    return "reject" if "suspicious" in question.lower() else "approve"


# The iterate notebook factored its streaming loop out into
# `stream_until_end_turn`, and most other notebooks just import it.
# This one doesn't, because every decision the agent makes is a
# custom tool call, which means the session keeps going idle with
# `stop_reason.type == "requires_action"` and
# `stop_reason.event_ids` pointing at the `agent.custom_tool_use`
# events waiting for a response. We POST a `user.custom_tool_result`
# for each, let the session resume, and eventually break on a
# `session.status_idle` that arrives with `end_turn`. The helper
# only knows how to exit on `end_turn`, so we need the full loop
# here.
decisions = {}  # receipt_id -> final decision record
tool_use_events = {}
responded_to = set()  # event_ids we've already replied to
print("=== Part A: streaming ===")
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Read /mnt/session/uploads/policy.yaml and "
                            "/mnt/session/uploads/receipts.jsonl. Process "
                            "all 12 receipts. For each receipt, make "
                            "exactly one decide() or escalate() call and "
                            "then move on to the next. When every receipt "
                            "has been processed once, stop."
                        ),
                    }
                ],
            }
        ],
    )
    # Note on the responded_to set: when an agent emits more than 5
    # parallel custom tool calls, the server returns
    # `stop_reason.event_ids` as a sliding window of the next 5
    # pending. Each status_idle we observe in the stream has that
    # window pinned at the moment the event was emitted, but by the
    # time we iterate to the next status_idle event, the server has
    # already advanced past the events we just responded to. So we
    # need to dedupe across status_idle events to avoid double-
    # responding to the same custom tool call (which 400s).
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded_to:
                        continue
                    tool_ev = tool_use_events[event_id]
                    name, args = tool_ev.name, tool_ev.input
                    receipt_id = args["receipt_id"]
                    if name == "decide":
                        decisions[receipt_id] = {"lane": args["action"], **args}
                        result = {"recorded": True}
                    elif name == "escalate":
                        human = simulate_human_review(receipt_id, args["question"])
                        decisions[receipt_id] = {
                            "lane": "escalated",
                            "human_decision": human,
                            **args,
                        }
                        result = {"human_decision": human}
                    else:
                        result = {"error": f"unknown tool {name}"}
                    client.beta.sessions.events.send(
                        session_id=session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded_to.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, session.id)

lanes = Counter(d["lane"] for d in decisions.values())
print(f"\n{len(decisions)} decisions: {dict(lanes)}")

client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## B부: 프로덕션을 위한 웹훅

로컬 스트리밍 패턴은 개발 중에는 잘 동작하지만, 사람이 고민하는 동안 HTTP 연결을 열어 두기 때문에 확장성이 좋지 않습니다. 프로덕션 패턴에서는 대신 콘솔에 `session.status_idled`에 반응하는 웹훅을 등록합니다. 이 이벤트는 에이전트가 작업을 마쳤거나 도구 결과를 기다리고 있다는 신호입니다. 여러분의 서버는 이벤트를 살펴보고, 대기 중인 에스컬레이션을 검토자 앞에 올려 두고, 사람이 끝내는 시점에 `user.custom_tool_result`를 POST로 돌려보냅니다. 여러분 쪽에 오래 유지되는 연결이 없습니다.

operate 노트북은 웹훅 설정 전체를 처음부터 끝까지 다룹니다. 콘솔 등록, HMAC 서명 검증, FastAPI 핸들러, 그리고 `events.send`로 돌려보내는 왕복까지입니다. 에이전트에 응답하는 코드는 위 A부와 동일하고, 트리거만 달라집니다(스트리밍 풀 대신 웹훅 푸시).